## 🔄 Compare Results Across All Datasets

After running experiments on multiple datasets, compare their performance.

In [1]:
# Compare results across all datasets
import json
from pathlib import Path
import pandas as pd

exp_base = Path('experiments')
datasets = ['DJI', 'IXIC', 'NYSE']
comparison_data = []

for dataset in datasets:
    # Find experiment directories for this dataset
    exp_dirs = [d for d in exp_base.glob(f'SAMBA_{dataset}_*') if d.is_dir()]
    
    if exp_dirs:
        # Get the latest experiment for this dataset
        latest_exp = max(exp_dirs, key=lambda x: x.stat().st_mtime)
        summary_file = latest_exp / 'experiment_summary.json'
        
        if summary_file.exists():
            with open(summary_file, 'r') as f:
                summary = json.load(f)
            
            stats = summary['statistics']
            best_run = summary['best_run']
            
            comparison_data.append({
                'Dataset': dataset,
                'Best MAE': best_run['test_mae'],
                'Best RMSE': best_run['test_rmse'],
                'Best IC': best_run.get('test_ic', 0),
                'Best RIC': best_run.get('test_ric', 0),
                'Avg MAE': stats['test_mae']['mean'],
                'Avg RMSE': stats['test_rmse']['mean'],
                'Avg IC': stats['test_ic']['mean'],
                'Avg RIC': stats['test_ric']['mean'],
                'Std MAE': stats['test_mae']['std'],
                'Std RMSE': stats['test_rmse']['std'],
                'Std IC': stats['test_ic']['std'],
                'Std RIC': stats['test_ric']['std'],
                'Runs': stats['num_runs']
            })

if comparison_data:
    df = pd.DataFrame(comparison_data)
    print("="*110)
    print("📊 PERFORMANCE COMPARISON ACROSS DATASETS")
    print("="*110)
    print(df.to_string(index=False))
    print("="*110)
    
    # Find best performing dataset
    best_dataset = df.loc[df['Best MAE'].idxmin(), 'Dataset']
    print(f"\n🏆 Best performing dataset (lowest MAE): {best_dataset}")
    
    # Find best IC dataset
    best_ic_dataset = df.loc[df['Best IC'].idxmax(), 'Dataset']
    print(f"🏆 Best IC (highest correlation): {best_ic_dataset}")
else:
    print("❌ No experiment results found. Run experiments first!")

📊 PERFORMANCE COMPARISON ACROSS DATASETS
Dataset  Best MAE  Best RMSE  Best IC  Best RIC  Avg MAE  Avg RMSE   Avg IC  Avg RIC  Std MAE  Std RMSE   Std IC  Std RIC  Runs
    DJI  0.035560   0.037811 0.957375  0.949498 0.022195  0.024544 0.964698 0.963900 0.010135  0.010089 0.006597 0.006321    20
   IXIC  0.026991   0.028420 0.947177  0.943859 0.035036  0.036413 0.827500 0.829526 0.024998  0.024682 0.405993 0.405863    20
   NYSE  0.014432   0.017399 0.957323  0.958062 0.015021  0.018375 0.959738 0.961906 0.005545  0.006404 0.020982 0.016673    20

🏆 Best performing dataset (lowest MAE): NYSE
🏆 Best IC (highest correlation): DJI


## 📊 Analyze Results

After running experiments, analyze and visualize the results.

In [2]:
# Find the latest experiment directory
import os
from pathlib import Path

exp_base = Path('experiments')
if exp_base.exists():
    # Get all SAMBA experiment directories, sorted by modification time
    exp_dirs = [d for d in exp_base.iterdir() if d.is_dir() and d.name.startswith('SAMBA_')]
    if exp_dirs:
        latest_exp = max(exp_dirs, key=lambda x: x.stat().st_mtime)
        print(f"📁 Latest experiment: {latest_exp.name}")
        print(f"📊 Analyzing results from: {latest_exp}")
        
        # Analyze results
        !python analyze_results.py --exp_dir "{latest_exp}" --regenerate_plots --regenerate_reports
    else:
        print("❌ No SAMBA experiment directories found")
else:
    print("❌ Experiments directory not found. Run experiments first!")

📁 Latest experiment: SAMBA_NYSE_20251012_145600
📊 Analyzing results from: experiments\SAMBA_NYSE_20251012_145600

📊 Analysis of: SAMBA_NYSE_20251012_145600

🏆 Best Run (by val_loss): Seed 14
──────────────────────────────────────────────────────────────────────
   Validation Loss: 103.1671
   Test MAE: 0.0144
   Test RMSE: 0.0174
   Best Epoch: 393
   Training Time: 15.74 minutes
   Status: Early stopped

📈 Statistics Across 20 Runs:
──────────────────────────────────────────────────────────────────────
   Successful Runs: 20/20

   Validation Loss:
      Mean: 112.3600 ± 5.5412
      Min:  103.1671
      Max:  125.5774

   Test MAE:
      Mean: 0.0150 ± 0.0055
      Min:  0.0067
      Max:  0.0263

   Test RMSE:
      Mean: 0.0184 ± 0.0064
      Min:  0.0090
      Max:  0.0325

   Convergence:
      Avg Best Epoch: 551.8 ± 258.8

   Training Time:
      Total: 7.03 hours
      Mean:  21.10 minutes/run

🌟 Top 10 Runs:
────────────────────────────────────────────────────────────────────

In [3]:
!python per_run_metrics_plot.py

🔍 Collecting metrics from all experiment runs...
📊 Processing DJI: Found 20 runs in SAMBA_DJI_20251012_011921
📊 Processing IXIC: Found 20 runs in SAMBA_IXIC_20251012_110314
📊 Processing NYSE: Found 20 runs in SAMBA_NYSE_20251012_145600
✅ Collected data from 60 runs across 3 datasets
✅ Box plot visualization saved as: experiments/per_run_metrics_boxplot.png
✅ Line plot visualization saved as: experiments/per_run_metrics_lineplot.png
Figure(1500x1200)
Figure(1800x600)
Figure(640x480)

📊 Summary Statistics:
         RMSE                      ...       RIC                    
        count      mean       std  ...       50%       75%       max
Dataset                            ...                              
DJI      20.0  0.024544  0.010351  ...  0.963498  0.968033  0.976165
IXIC     20.0  0.036413  0.025323  ...  0.938159  0.954869  0.967536
NYSE     20.0  0.018375  0.006570  ...  0.965785  0.968850  0.975894

[3 rows x 24 columns]
✅ Summary statistics saved as: experiments/per_run_me

d:\Google Drive\Resources_MQ\Session_2_2025\COMP8240 Application of Data Science\Project\Samba2\SAMBA\per_run_metrics_plot.py:89: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
d:\Google Drive\Resources_MQ\Session_2_2025\COMP8240 Application of Data Science\Project\Samba2\SAMBA\per_run_metrics_plot.py:89: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
d:\Google Drive\Resources_MQ\Session_2_2025\COMP8240 Application of Data Science\Project\Samba2\SAMBA\per_run_metrics_plot.py:89: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
